# Vaccine Processing Workflow

This notebook reproduces the final dataset derived from `COVID-19_Vaccinations_in_the_United_States,Jurisdiction_20260329.csv`.

It performs these steps:
1. Load the raw CSV.
2. Keep rows from years 2021-2025.
3. Sort by `Location` and `Date`.
4. Rename vaccination-related columns.
5. Rebuild `vac2022_1` as the sum of the manufacturer-specific second booster columns.
6. Write the final outputs to `vaccination_2021-2025.csv` and `vaccination_2021-2025`.


In [ ]:
import csv
from datetime import datetime
from pathlib import Path
import shutil

RAW_PATH = Path("COVID-19_Vaccinations_in_the_United_States,Jurisdiction_20260329.csv")
OUTPUT_CSV = Path("vaccination_2021-2025.csv")
OUTPUT_COPY = Path("vaccination_2021-2025")

PREFIX_RENAMES = [
    ("Series_Complete", "vac2021_1"),
    ("Additional_Doses", "vac2021_2"),
    ("Second_Booster", "vac2022_1"),
    ("Administered_Bivalent", "vac2022_2"),
    ("Bivalent_Booster", "vac2022_2"),
]

VAC2022_1_SOURCE_COLS = [
    "Second_Booster_Janssen",
    "Second_Booster_Moderna",
    "Second_Booster_Pfizer",
    "Second_Booster_Unk_Manuf",
]


def parse_date(value: str):
    for fmt in ("%m/%d/%Y", "%m/%d/%y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass
    raise ValueError(f"Unsupported date format: {value!r}")


def format_short_date(value):
    return f"{value.month}/{value.day}/{value.year % 100:02d}"


def parse_int(value: str):
    value = (value or "").strip()
    if not value:
        return None
    return int(value.replace(",", ""))


with RAW_PATH.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = []

    for row in reader:
        date_obj = parse_date(row["Date"])
        if not (2021 <= date_obj.year <= 2025):
            continue

        vac2022_1_parts = [parse_int(row[col]) for col in VAC2022_1_SOURCE_COLS]
        if all(part is None for part in vac2022_1_parts):
            vac2022_1_value = ""
        else:
            vac2022_1_value = f"{sum(part or 0 for part in vac2022_1_parts):,}"

        processed_row = dict(row)
        processed_row["Date"] = format_short_date(date_obj)
        processed_row["Second_Booster"] = vac2022_1_value
        rows.append((processed_row["Location"], date_obj, processed_row))

renamed_fieldnames = []
for name in fieldnames:
    renamed = name
    for old, new in PREFIX_RENAMES:
        renamed = renamed.replace(old, new)
    renamed_fieldnames.append(renamed)

rows.sort(key=lambda item: (item[0], item[1]))

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=renamed_fieldnames)
    writer.writeheader()
    for _, _, row in rows:
        renamed_row = {}
        for old_name, new_name in zip(fieldnames, renamed_fieldnames):
            renamed_row[new_name] = row[old_name]
        writer.writerow(renamed_row)

shutil.copyfile(OUTPUT_CSV, OUTPUT_COPY)

print(f"Wrote {len(rows):,} rows to {OUTPUT_CSV}")
print(f"Copied final output to {OUTPUT_COPY}")
print(f"First date/location: {rows[0][2]['Date']} {rows[0][2]['Location']}")
print(f"Last date/location: {rows[-1][2]['Date']} {rows[-1][2]['Location']}")
